# Homework 4: Club Chatbot
A topic-locked LangGraph tutor with Supabase checkpointing and a Gradio interface. Run this notebook in Google Colab after adding `OPENAI_API_KEY` and `SUPABASE_DB_URL` to Colab Secrets.

## Installation & Secrets
Install the notebook dependencies, then load both credentials from Colab Secrets. No secret is displayed or stored in notebook code.

In [ ]:
!pip -q install langgraph langchain langchain-openai langgraph-checkpoint-postgres "psycopg[binary]" gradio

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
SUPABASE_DB_URL = userdata.get("SUPABASE_DB_URL")

if not os.environ["OPENAI_API_KEY"] or not SUPABASE_DB_URL:
    raise ValueError("Add OPENAI_API_KEY and SUPABASE_DB_URL to Colab Secrets and enable notebook access.")

## State & Prompt Guard
The selected club is part of graph state. The system instruction is rebuilt for every turn, so changing clubs updates the guard while the same conversation history remains available.

In [ ]:
from typing import Annotated, TypedDict, Any
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import display, Image

MODEL_NAME = "openai:gpt-4o-mini"
CLUBS = ["Astronomy", "Cooking", "Football", "World Geography", "Python Programming"]
model = init_chat_model(MODEL_NAME)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str

SYSTEM_PROMPT = """You are a kind, helpful tutor at Curious Minds, an after-school center for teenagers.
The child is currently in the {topic} club. You are locked to this club and must ONLY answer questions related to {topic}.
If the child asks about anything outside {topic}, politely refuse, explicitly say that {topic} is the current club, and suggest switching clubs using the dropdown.
This restriction also applies to sneaky framing and follow-up questions about an earlier club topic that appears in the conversation history.
Use the current club topic, not an older topic in the history, to decide what is allowed.
"""

def chatbot(state: ChatState) -> dict[str, list[AnyMessage]]:
    system_message = SystemMessage(content=SYSTEM_PROMPT.format(topic=state["topic"]))
    response = model.invoke([system_message] + state["messages"] )
    return {"messages": [response]}

## Supabase Checkpointer & Graph
The Session pooler lets Colab connect to Supabase. `PostgresSaver` persists graph state by `thread_id`, so conversations survive notebook runtime restarts.

In [ ]:
import psycopg
from psycopg.rows import dict_row
from langgraph.checkpoint.postgres import PostgresSaver

conn = psycopg.connect(
    userdata.get("SUPABASE_DB_URL"),
    autocommit=True,
    row_factory=dict_row,
    prepare_threshold=0,
)
checkpointer = PostgresSaver(conn)
checkpointer.setup()

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=checkpointer)
display(Image(graph.get_graph().draw_mermaid_png()))

## Standalone Graph Test
This example uses one thread to demonstrate a Cooking answer, an off-topic refusal, a club switch, and a refusal of a follow-up about the previous club. The checkpoint state keeps the message history and latest topic together.

In [ ]:
config = {"configurable": {"thread_id": "test-1"}}

def ask(text: str, topic: str) -> None:
    result = graph.invoke(
        {"messages": [HumanMessage(content=text)], "topic": topic},
        config,
    )
    print(f"[{topic}] {text}\n -> {result['messages'][-1].content}\n")

ask("How do I make pancakes?", "Cooking")
ask("Who won the 2022 World Cup?", "Cooking")
ask("Who won the 2022 World Cup?", "Football")
ask("And how long do I fry each pancake?", "Football")
saved_state = graph.get_state(config).values
print("Saved topic:", saved_state["topic"])
print("Messages saved:", len(saved_state["messages"]))

## Gradio UI
The interface stores only the active thread ID as UI state. The list of saved conversations is rebuilt from checkpoint records in Supabase, and reopening a conversation restores its messages and selected club.

In [ ]:
import uuid
import gradio as gr

def messages_for_ui(messages: list[AnyMessage]) -> list[dict[str, str]]:
    return [
        {"role": "user" if isinstance(message, HumanMessage) else "assistant", "content": str(message.content)}
        for message in messages
        if isinstance(message, (HumanMessage, AIMessage))
    ]

def saved_conversation_choices() -> list[tuple[str, str]]:
    # Read checkpoint records from Postgres; do not keep a separate conversation store.
    newest_by_thread: dict[str, Any] = {}
    for checkpoint_tuple in checkpointer.list(None):
        thread_id = checkpoint_tuple.config["configurable"]["thread_id"]
        if thread_id not in newest_by_thread:
            newest_by_thread[thread_id] = checkpoint_tuple.checkpoint

    choices = []
    for thread_id, checkpoint in newest_by_thread.items():
        values = checkpoint.get("channel_values", {})
        messages = values.get("messages", [])
        topic = values.get("topic", "Unknown club")
        first_question = next((str(m.content) for m in messages if isinstance(m, HumanMessage)), "New conversation")
        label = f"{topic} — {first_question[:35]}"
        choices.append((label, thread_id))
    return choices

def send_message(message: str, history: list[dict], topic: str, thread_id: str):
    if not message or not message.strip():
        return history, "", gr.update(choices=saved_conversation_choices()), thread_id
    result = graph.invoke(
        {"messages": [HumanMessage(content=message)], "topic": topic},
        {"configurable": {"thread_id": thread_id}},
    )
    updated_history = messages_for_ui(result["messages"])
    return updated_history, "", gr.update(choices=saved_conversation_choices(), value=thread_id), thread_id

def new_conversation():
    thread_id = str(uuid.uuid4())
    return [], thread_id, "Astronomy", gr.update(choices=saved_conversation_choices(), value=None)

def load_conversation(thread_id: str):
    if not thread_id:
        return [], "Astronomy", ""
    state = graph.get_state({"configurable": {"thread_id": thread_id}}).values
    return messages_for_ui(state.get("messages", [])), state.get("topic", "Astronomy"), thread_id

with gr.Blocks() as app:
    gr.Markdown("# Curious Minds Club Chatbot")
    active_thread = gr.State(str(uuid.uuid4()))
    with gr.Row():
        topic_dropdown = gr.Dropdown(choices=CLUBS, value="Astronomy", label="Club")
        saved_dropdown = gr.Dropdown(choices=saved_conversation_choices(), label="Saved conversations")
        new_button = gr.Button("+ New conversation")
    chat = gr.Chatbot(type="messages", label="Conversation")
    message_box = gr.Textbox(label="Message", placeholder="Ask something about this club...")
    message_box.submit(
        send_message,
        [message_box, chat, topic_dropdown, active_thread],
        [chat, message_box, saved_dropdown, active_thread],
    )
    new_button.click(
        new_conversation,
        [],
        [chat, active_thread, topic_dropdown, saved_dropdown],
    )
    saved_dropdown.change(
        load_conversation,
        [saved_dropdown],
        [chat, topic_dropdown, active_thread],
    )

app.launch()